Import dei pacchetti necessari

In [ ]:
# Installazione librerie (da eseguire in una cella Colab)
!pip install datasets transformers[torch] tokenizers evaluate rouge_score sentencepiece huggingface_hub nltk


In [ ]:
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)
import evaluate


Caricamento dei dati

In [ ]:
# Esempio: caricamento di un dataset JSON con Hugging Face
data_files = {"train": "/content/Semantic/dataset_en.json"}  # sostituire con il percorso corretto
dataset = load_dataset("json", data_files=data_files)
print(dataset)


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['non_inclusiva', 'inclusiva'],
        num_rows: 4980
    })
})


Definizione del modello

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
model.to("cuda")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:
MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Preprocessing: tokenizzazione e creazione del dataset

In [ ]:
from datasets import DatasetDict
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
import torch

# --- 1) usa le colonne REALI del tuo dataset_en.json (sono queste) ---
SOURCE_COL = "non_inclusiva"
TARGET_COL = "inclusiva"

# --- 2) split train/validation ---
raw = dataset["train"]
spl = raw.train_test_split(test_size=0.1, seed=42)
splits = DatasetDict({"train": spl["train"], "validation": spl["test"]})

# --- 3) preprocess/tokenize (sicuro per T5/FlanT5) ---
prefix = "Rewrite the sentence using inclusive language: "

def preprocess_function(examples):
    inputs = [prefix + str(x) for x in examples[SOURCE_COL]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)

    targets = [str(x) for x in examples[TARGET_COL]]
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = splits.map(
    preprocess_function,
    batched=True,
    remove_columns=splits["train"].column_names
)

# --- 4) FILTRO CRITICO: rimuove esempi con labels vuote o solo PAD (causano loss NaN) ---
def has_valid_labels(ex):
    labs = ex["labels"]
    if labs is None or len(labs) == 0:
        return False
    # deve esistere almeno 1 token diverso dal PAD
    return any(t != tokenizer.pad_token_id for t in labs)

tokenized_datasets = tokenized_datasets.filter(has_valid_labels)

# --- 5) collator corretto: padding labels a -100 (ignorati nella loss) ---
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

# --- 6) SANITY CHECK: se qui vedi zeri -> ancora labels “vuote” -> NaN ---
batch = next(iter(DataLoader(tokenized_datasets["validation"], batch_size=4, collate_fn=data_collator)))
print("Non-masked label tokens per sample:", (batch["labels"] != -100).sum(dim=1).tolist())

# --- 7) test loss secca (prima di trainare) ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
for k in batch:
    batch[k] = batch[k].to(device)
with torch.no_grad():
    out = model(**batch)
print("Sanity loss:", out.loss.item(), "isfinite:", torch.isfinite(out.loss).item())


Map:   0%|          | 0/4482 [00:00<?, ? examples/s]

Map:   0%|          | 0/498 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4482 [00:00<?, ? examples/s]

Filter:   0%|          | 0/498 [00:00<?, ? examples/s]

Non-masked label tokens per sample: [16, 24, 20, 17]
Sanity loss: 3.726625442504883 isfinite: True


Addestramento con Seq2SeqTrainer

In [ ]:
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.array(preds)

    if preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)
    else:
        # Se sono già sequenze ma in float/NaN: ripulisci e cast
        preds = np.nan_to_num(
            preds,
            nan=tokenizer.pad_token_id,
            posinf=tokenizer.pad_token_id,
            neginf=tokenizer.pad_token_id
        )
        preds = np.rint(preds).astype(np.int64)

    # labels: sostituisci -100 con pad per decodifica
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # clip di sicurezza per evitare id fuori range
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(preds.tolist(), skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels.tolist(), skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: v * 100 for k, v in result.items()}


training_args = Seq2SeqTrainingArguments(
    output_dir="./model_out",
    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,   

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    learning_rate=5e-5,             
    warmup_ratio=0.05,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,

    logging_strategy="steps",
    logging_steps=50,
    report_to="none",

    fp16=False,                     
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [ ]:
trainer.train()
print(trainer.evaluate())

pred = trainer.predict(tokenized_datasets["validation"])

preds = pred.predictions
if isinstance(preds, tuple):
    preds = preds[0]

preds = np.array(preds)

# se sono logits (N, T, V) -> argmax
if preds.ndim == 3:
    preds_ids = np.argmax(preds, axis=-1)
else:
    # se sono già sequenze ma non int -> cast sicuro
    preds_ids = np.nan_to_num(
        preds,
        nan=tokenizer.pad_token_id,
        posinf=tokenizer.pad_token_id,
        neginf=tokenizer.pad_token_id
    )
    preds_ids = np.rint(preds_ids).astype(np.int64)

# clamp per evitare id fuori vocabolario
preds_ids = np.clip(preds_ids, 0, tokenizer.vocab_size - 1)

# decode sample + reference sample
print("Pred sample:", tokenizer.decode(preds_ids[0].tolist(), skip_special_tokens=True))

labels = np.where(pred.label_ids != -100, pred.label_ids, tokenizer.pad_token_id)
print("Ref  sample:", tokenizer.decode(labels[0].tolist(), skip_special_tokens=True))


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,1.584900,1.535459,37.691926,15.332805,33.323582,33.283497
2,1.488100,1.508244,38.969188,15.951764,34.741068,34.665621
3,1.430600,1.504429,39.099335,15.742305,34.712810,34.670264


{'eval_loss': 1.504428744316101, 'eval_rouge1': 39.09933498263237, 'eval_rouge2': 15.742304776958626, 'eval_rougeL': 34.71280952301304, 'eval_rougeLsum': 34.67026388632106, 'eval_runtime': 74.5168, 'eval_samples_per_second': 6.683, 'eval_steps_per_second': 1.677, 'epoch': 3.0}
Pred sample: Sexual orientation does not determine a person's aptitude for dance.
Ref  sample: Stereotypes about skills or hobbies based on sexual orientation are inaccurate.


Salvataggio del modello fine-tuned

In [ ]:
SAVE_DIR = "./model_inclusive_rewriter"

trainer.save_model(SAVE_DIR)          # salva modello + config
tokenizer.save_pretrained(SAVE_DIR)   # salva tokenizer


('./model_inclusive_rewriter/tokenizer_config.json',
 './model_inclusive_rewriter/special_tokens_map.json',
 './model_inclusive_rewriter/spiece.model',
 './model_inclusive_rewriter/added_tokens.json',
 './model_inclusive_rewriter/tokenizer.json')

Valutazione finale

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

loader = DataLoader(tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator)

all_preds, all_refs = [], []

for batch in loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        gen_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=64,
            num_beams=4
        )

    preds_text = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

    labels = batch["labels"].cpu().numpy()
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    refs_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    all_preds.extend(preds_text)
    all_refs.extend(refs_text)

rouge_results = rouge.compute(predictions=all_preds, references=all_refs, use_stemmer=True)
print({k: v * 100 for k, v in rouge_results.items()})


{'rouge1': np.float64(39.09933498263237), 'rouge2': np.float64(15.742304776958626), 'rougeL': np.float64(34.71280952301304), 'rougeLsum': np.float64(34.67026388632106)}


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

examples = [
    "Women can't do anything except cook.",
    "Disabled people are bad at soccer.",
    "Blonde people are stupid.",
]

inputs = [prefix + s for s in examples]
enc = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    gen = model.generate(
        **enc,
        max_new_tokens=64,
        num_beams=4,
        do_sample=False
    )

outs = tokenizer.batch_decode(gen, skip_special_tokens=True)

for s, o in zip(examples, outs):
    print("Input (non-inclusive):", s)
    print("Output (inclusive):   ", o)
    print()


Input (non-inclusive): Women can't do anything except cook.
Output (inclusive):    Skills and abilities vary among individuals, regardless of gender.

Input (non-inclusive): Disabled people are bad at soccer.
Output (inclusive):    Athletic ability is individual and not determined by disability.

Input (non-inclusive): Blonde people are stupid.
Output (inclusive):    Hair color does not determine a person's intelligence.

